# 記事に似た記事を推薦する（関連記事、重複除去・多様化・新しさ）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_search/related_articles.ipynb)

NLP クックブック 22-15 ／ 検索

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

ニュースサイトの記事の下に「関連記事」を 5 件出したい。埋め込みで似た記事を探すだけでなく、ほぼ同じ内容の記事を除き、似た記事ばかりにならないようにし、新しさも考慮して、その効果を数値で比べたい。

In [1]:
import io, tarfile, urllib.request
import numpy as np
import pandas as pd

def load_livedoor():
    """livedoor ニュースコーパス (株式会社ロンウイット配布、記事は CC BY-ND 2.1 JP)。1 行目 URL、2 行目 日付、3 行目 タイトル"""
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "date": lines[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert("Asia/Tokyo")
print(len(df), "記事", df["date"].min().date(), "〜", df["date"].max().date())

7367 記事 2000-03-01 〜 2012-09-04


In [2]:
from sentence_transformers import SentenceTransformer

# 記事 = タイトル + 本文の冒頭 120 字。e5 は文同士の類似度 (対称なタスク) では両側に "query: " を付ける
model = SentenceTransformer("intfloat/multilingual-e5-small")   # MIT ライセンス
model.max_seq_length = 128                                   # これを超えるトークンは切られる。120 字ならほぼ全記事が収まる
body = df["text"].str.split().str.join(" ").str[:120]
emb = model.encode(("query: " + df["title"] + "\n" + body).tolist(),
                   normalize_embeddings=True, batch_size=64, show_progress_bar=False)   # GPU (T4) なら 1 分以内。CPU だけでも動くが Colab では 10〜15 分

S = emb @ emb.T
np.fill_diagonal(S, -np.inf)                                 # 自分自身は推薦しない
C = 50                                                       # 類似度の上位 50 件を候補にして、その中から選ぶ
top = np.argpartition(-S, C, axis=1)[:, :C]                 # 上位 50 件 (順不同) だけを取り出し、
cand = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, axis=1), axis=1), axis=1)   # 類似度順に並べる
del top
print(emb.shape, "候補", cand.shape)

(7367, 384) 候補 (7367, 50)


In [3]:
from sklearn.feature_extraction.text import CountVectorizer

# 「ほぼ同じ内容」は埋め込みではなく、本文の文字 5-gram の Jaccard 係数で判定する (テンプレート記事や重複掲載を拾う)
X = CountVectorizer(analyzer="char", ngram_range=(5, 5), binary=True, dtype=np.float32).fit_transform(
    df["text"].str.replace(r"\s+", "", regex=True))
size = np.asarray(X.sum(axis=1)).ravel()
N = len(df)
JB = np.empty((N, C + 1, C + 1), dtype=np.float32)          # 記事ごとに [元の記事, 候補 50 件] 同士の Jaccard 係数
for i in range(N):                                           # Colab で 1〜2 分
    b = np.r_[i, cand[i]]
    inter = (X[b] @ X[b].T).toarray()
    JB[i] = inter / (size[b][:, None] + size[b][None, :] - inter)

labels = df["label"].to_numpy()
age = (df["date"].max() - df["date"]).dt.days.to_numpy()     # コーパスの最終日を「今日」とみなした記事の古さ
fresh = 0.5 ** (age / 180)                                   # 半減期 180 日の新しさ (今日 = 1、180 日前 = 0.5)

def recommend(i, k=5, dup=None, lam=1.0, beta=0.0, gamma=0.0):
    """dup: Jaccard 係数がこれを超える記事を除く / lam: MMR の重み (1 なら類似度順)
    beta: 新しさのボーナス / gamma: 同じカテゴリのボーナス
    返り値: 推薦する記事の番号と、候補 50 件の中での位置 (JB を引くのに使う)"""
    c, J = cand[i], JB[i]
    rel = S[i, c] + beta * fresh[c] + gamma * (labels[c] == labels[i])
    ok = np.ones(C, dtype=bool) if dup is None else J[0, 1:] <= dup
    red, chosen = np.zeros(C), []
    for _ in range(k):
        score = np.where(ok, lam * rel - (1 - lam) * red, -np.inf)
        p = int(np.argmax(score))
        if not np.isfinite(score[p]):
            break
        chosen.append(p)
        ok[p] = False
        if dup is not None:
            ok &= J[p + 1, 1:] <= dup                        # 選んだ記事とほぼ同じ記事も除く
        red = np.maximum(red, emb[c] @ emb[c[p]])            # 選んだ記事との最大類似度 (MMR の罰則)
    chosen = np.array(chosen, dtype=int)
    return c[chosen], chosen

In [4]:
i = int(np.flatnonzero(df["title"].str.contains("土屋アンナ編"))[0])
print(f"元の記事: [{labels[i]}] {df['title'][i]}")
for name, kw in [("類似度の上位 5 件", {}), ("近重複を除く + MMR (λ=0.7)", {"dup": 0.5, "lam": 0.7})]:
    print(name)
    recs, pos = recommend(i, **kw)
    for j, p in zip(recs, pos):
        print(f"  類似度 {S[i, j]:.3f}  Jaccard {JB[i][0, p + 1]:.2f}  本文 {len(df['text'][j]):4d} 字  "
              f"{df['date'][j].date()} [{labels[j]}] {df['title'][j][:36]}")

元の記事: [peachy] 【スナップレポート】東京ガールズコレクション 2011 土屋アンナ編
類似度の上位 5 件
  類似度 0.965  Jaccard 0.63  本文   41 字  2011-09-16 [peachy] 【スナップレポート】東京ガールズコレクション 2011 藤井リナ編
  類似度 0.961  Jaccard 0.60  本文   40 字  2009-09-16 [peachy] 【スナップレポート】東京ガールズコレクション 2011 佐々木希編
  類似度 0.961  Jaccard 0.60  本文   40 字  2011-09-16 [peachy] 【スナップレポート】東京ガールズコレクション 2011 佐々木希編
  類似度 0.957  Jaccard 0.60  本文   40 字  2011-09-16 [peachy] 【スナップレポート】東京ガールズコレクション 2011 水原希子編
  類似度 0.923  Jaccard 0.04  本文  571 字  2011-09-13 [peachy] “モデル別で見る”東京ガールズコレクション 2011 スナップレポートN
近重複を除く + MMR (λ=0.7)
  類似度 0.923  Jaccard 0.04  本文  571 字  2011-09-13 [peachy] “モデル別で見る”東京ガールズコレクション 2011 スナップレポートN
  類似度 0.905  Jaccard 0.01  本文 1589 字  2011-03-11 [peachy] 東京ガールズコレクションで最もフィーチャーされた2人とは？
  類似度 0.892  Jaccard 0.00  本文  945 字  2011-12-30 [dokujo-tsushin] 【独女通信編集部】2011年記事まとめ　vol.2
  類似度 0.900  Jaccard 0.00  本文  805 字  2011-01-17 [livedoor-homme] 【最新キャンギャル＆レースクイーン写真集 東京オートサロン2011】  
  類似度 0.909  Jaccard 0.02  本文 1094 字  2011-09-13 [peachy] 東京ガールズコレクション 2011 A

In [5]:
def gini(counts):
    """推薦される回数の偏り (0 = 全記事が同じ回数、1 に近いほど一部の記事に集中)"""
    x = np.sort(counts)
    n = len(x)
    return float(((2 * np.arange(1, n + 1) - n - 1) * x).sum() / (n * x.sum()))

def evaluate(**kw):
    out = [recommend(i, **kw) for i in range(N)]
    recs = [r for r, _ in out]
    counts = np.bincount(np.concatenate(recs), minlength=N)
    def has_dup(i, pos):                                     # 元の記事と推薦した記事の組のどれかが Jaccard 0.5 超
        b = np.r_[0, pos + 1]
        return (JB[i][np.ix_(b, b)][np.triu_indices(len(b), 1)] > 0.5).any()
    return {
        "類似度": np.mean([S[i, r].mean() for i, r in enumerate(recs)]),
        "同カテゴリ率": np.mean([(labels[r] == labels[i]).mean() for i, r in enumerate(recs)]),
        "カバレッジ": (counts > 0).mean(),                    # 1 回以上推薦された記事の割合
        "Gini": gini(counts),
        "最多の推薦回数": counts.max(),                       # いちばん多く推薦された記事の回数
        "リスト内類似度": np.mean([(emb[r] @ emb[r].T)[np.triu_indices(len(r), 1)].mean() for r in recs]),
        "近重複を含む率": np.mean([has_dup(i, p) for i, (_, p) in enumerate(out)]),
        "平均の古さ(日)": np.mean([age[r].mean() for r in recs]),
    }

settings = {
    "類似度の上位 5 件": {},
    "+ 近重複を除く": {"dup": 0.5},
    "+ MMR (λ=0.7)": {"dup": 0.5, "lam": 0.7},
    "+ MMR (λ=0.5)": {"dup": 0.5, "lam": 0.5},
    "+ MMR (λ=0.7) + 新しさ (β=0.02)": {"dup": 0.5, "lam": 0.7, "beta": 0.02},
    "+ MMR (λ=0.7) + 新しさ (β=0.05)": {"dup": 0.5, "lam": 0.7, "beta": 0.05},
    "+ MMR (λ=0.7) + 同カテゴリ (γ=0.02)": {"dup": 0.5, "lam": 0.7, "gamma": 0.02},
}
pd.DataFrame({name: evaluate(**kw) for name, kw in settings.items()}).T.round(3)

,類似度,同カテゴリ率,カバレッジ,Gini,最多の推薦回数,リスト内類似度,近重複を含む率,平均の古さ(日)
類似度の上位 5 件,0.903,0.775,0.913,0.451,52.0,0.892,0.055,280.188
+ 近重複を除く,0.903,0.773,0.912,0.454,52.0,0.891,0.000,279.741
+ MMR (λ=0.7),0.900,0.732,0.970,0.344,23.0,0.876,0.000,280.483
+ MMR (λ=0.5),0.890,0.650,0.982,0.293,18.0,0.857,0.000,281.966
+ MMR (λ=0.7) + 新しさ (β=0.02),0.898,0.709,0.906,0.503,66.0,0.875,0.000,164.974
+ MMR (λ=0.7) + 新しさ (β=0.05),0.894,0.674,0.727,0.701,134.0,0.873,0.000,97.511
+ MMR (λ=0.7) + 同カテゴリ (γ=0.02),0.899,0.955,0.968,0.348,26.0,0.878,0.000,284.044


- 流れは「埋め込みの類似度で候補を 50 件に絞る → ほぼ同じ内容の記事を除く → MMR（この章の「検索結果の重複を減らして多様にする (MMR)」）で選ぶ → 新しさやカテゴリのボーナスを足す」。記事の埋め込みと推薦リストは記事の公開時やバッチで計算しておき、表示のときは引くだけにする。ここでは全記事の類似度行列 (7367 × 7367) を作ったが、件数が多いときはこの章の「FAISS で大量のベクトルから高速に近傍検索する」で候補を取る。埋め込みは本文の冒頭 120 字までにした。300 字にしても `max_seq_length=128` を超えた分は切られ、128 より長くすると時間がかかる。
- 例の上位 4 件は、本文が 40 字ほどしかない同じ形式の写真記事（同じ連載の別のモデルの回）で、テンプレートの共通部分のために Jaccard 係数が 0.60〜0.63 と高い。本当の重複掲載は、同じタイトルで日付だけが違う「佐々木希編」の 2 本である。この例では埋め込みの類似度でも近重複 (0.957〜0.965) と別の記事 (0.923) は分かれているが、Jaccard 係数（0.60 以上と 0.04）の方が差がはっきりする。近重複を含む推薦リストは、類似度の上位 5 件では全記事の 5.5% だった。しきい値 0.5 は、短い記事では連載の各回まで除く。連載の各回は除くより「連載の他の回」として別枠に出す方がよいこともある。大量の文書ならデータの収集と入出力の章の「MinHash と LSH で大量文書からほぼ重複を見つける」を使う。
- 類似度の上位だけでは推薦が一部の記事に偏り（Gini 係数 0.451、最も多く推薦された記事は 52 回）、1 回も推薦されない記事が 1 割近く残る（カバレッジ 0.913）。MMR (λ=0.7) にすると、カバレッジは 0.970、Gini 係数は 0.344、最多の回数は 23 回になり、元の記事との平均類似度は 0.903 から 0.900 とわずかしか下がらない。ただし例の 3・4 件目（独女通信の年間まとめ、東京オートサロンの写真集）のように、関係の薄い記事も入る。λ=0.5 では同カテゴリ率が 0.650 まで下がる。
- 新しさのボーナスを強くすると (β=0.05)、MMR (λ=0.7) だけのときと比べて、推薦される記事の平均の古さは 280 日から 98 日になるが、カバレッジは 0.970 から 0.727 に下がり、最多の回数は 134 回になる（新しい記事に推薦が集まり、古い記事は推薦されなくなる）。時期の決まった話題では、今日からの古さより元の記事の日付との近さを使う方が自然なこともある。コーパスの日付には誤りらしいものがある（例の 2011 年の東京ガールズコレクションの記事に 2009 年の日付）ので、日付を使う前に分布を確かめる。
- 同カテゴリ率は関連度の手軽な代理だが、カテゴリをスコアに入れると（γ=0.02 で 0.955）当然上がるので、スコアに使った情報で評価してはいけない。これらのオフラインの指標はトレードオフを見るためのもので、最終的にはクリック率や回遊率を A/B テストで比べる。
- livedoor ニュースコーパスは株式会社ロンウイットが配布しているもので、記事は CC BY-ND 2.1 JP（改変禁止）、`intfloat/multilingual-e5-small` は MIT ライセンス。